PLEASE REFER TO APPENDIX ON HOW TO RUN THE CODE, CHANGE THE FILE NAMES ACCORDINGLY IN ORDER FOR THE DATASETS TO BE IMPORTED ON YOUR SYSTEM. YOU CAN LOCATE THE WHEREABOUTS OF THE CSV'S BY CHECKING THE PATHWAY WHERE THE CODE IS TRYING TO IMPORT


USE localized_url function if there's an error with the imports

In [ ]:
import os
def localized_url(url):

    current_dir = os.getcwd().split("statistical-analysis-and-modeling-of-football-data")[0] + "statistical-analysis-and-modeling-of-football-data"
    branch = url.split("statistical-analysis-and-modeling-of-football-data")[1]
    return current_dir + branch

In [83]:
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
import xgboost as xgb
import numpy as np
import pandas as pd
import random
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from tqdm import tqdm


import warnings
warnings.filterwarnings('ignore')  # Suppress Python warnings


random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)


In [ ]:
import os
def localized_url(url):

    current_dir = os.getcwd().split("statistical-analysis-and-modeling-of-football-data")[0] + "statistical-analysis-and-modeling-of-football-data"
    branch = url.split("statistical-analysis-and-modeling-of-football-data")[1]
    return current_dir + branch

In [ ]:
team_df= pd.read_csv(localized_url('/Users/yarkin/Desktop/Dissertation/statistical-analysis-and-modeling-of-football-data/Aggregation/Data/under_team_pivoted.csv'))
fifa_df= pd.read_csv(localized_url('/Users/yarkin/Desktop/Dissertation/statistical-analysis-and-modeling-of-football-data/Fifa/Data/fifa_stats_fully_processed.csv'))
elo_df = pd.read_csv(localized_url('/Users/yarkin/Desktop/Dissertation/statistical-analysis-and-modeling-of-football-data/ClubELO/Data/understat_team_elo_with_seasonal_elo.csv'))
aggregated_df = pd.read_csv(localized_url('/Users/yarkin/Desktop/Dissertation/statistical-analysis-and-modeling-of-football-data/Aggregation/Data/understat_team_aggregated.csv'))


In [85]:
aggregated_df = aggregated_df[['id', 'date', 'team_id', 'Shots On Target', 'DEEP', 'PPDA',
       'Shots On Target_mean',  'DEEP_mean',
       'PPDA_median', 'h_a']]

In [86]:
aggregated_df = aggregated_df[['id', 'date', 'team_id', 'Shots On Target', 'DEEP', 'PPDA',
       'Shots On Target_mean',  'DEEP_mean',
       'PPDA_median', 'h_a']]

In [87]:
aggregated_features_list= ['Shots On Target', 'DEEP', 'PPDA',
       'Shots On Target_mean',  'DEEP_mean',
       'PPDA_median']

In [88]:
elo_df = elo_df[['id', 'Elo_team_h',  'Elo_team_a', 'Seasonal_Elo_Home',
       'Seasonal_Elo_Away']].copy()
elo_df["ClubELO"] = elo_df["Elo_team_h"] - elo_df["Elo_team_a"]
elo_df["Seasonal_Elo"] = elo_df["Seasonal_Elo_Home"] - elo_df["Seasonal_Elo_Away"]
elo_df = elo_df[['id', 'ClubELO', 'Seasonal_Elo']].copy()

In [89]:
# Merge aggregated_df with elo_df, elo_df will be + if h_a == 'h' and - if h_a == 'a'
merged_df = pd.merge(aggregated_df, elo_df, on='id', how='left')
merged_df['ClubELO'] = np.where(merged_df['h_a'] == 'h', merged_df['ClubELO'], -merged_df['ClubELO'])
merged_df['Seasonal_Elo'] = np.where(merged_df['h_a'] == 'h', merged_df['Seasonal_Elo'], -merged_df['Seasonal_Elo'])

In [90]:
fifa_df.drop(columns=['yellow_card', 'red_card', 'key_passes', 'assists', 'xG',
       'xGBuildup'], inplace=True)

In [91]:
fifa_df.columns

Index(['id', 'Overall', 'Finishing_adj', 'Shot Power_adj', 'Long Shots_adj',
       'Heading_adj', 'Curve_adj', 'FK Acc._adj', 'Penalties_adj',
       'Short Pass_adj', 'Long Pass_adj', 'Stand Tackle_adj',
       'Interceptions_adj', 'Jumping_adj', 'Stamina_adj', 'Strength_adj',
       'Acceleration_adj', 'Agility_adj', 'Balance_adj', 'GK Handling_adj',
       'GK Kicking_adj', 'GK Positioning_adj', 'Reactions_adj',
       'Att. Position_adj', 'Vision_adj', 'Dribbling_adj',
       'Overall Passing_adj', 'Overall Shooting_adj', 'Overall Defence_adj',
       'Overall Physical_adj', 'Overall Goalkeeper_adj'],
      dtype='object')

In [92]:
fifa_stats = ['Overall', 'Finishing_adj', 'Shot Power_adj', 'Long Shots_adj',
       'Heading_adj', 'Curve_adj', 'FK Acc._adj', 'Penalties_adj',
       'Short Pass_adj', 'Long Pass_adj', 'Stand Tackle_adj',
       'Interceptions_adj', 'Jumping_adj', 'Stamina_adj', 'Strength_adj',
       'Acceleration_adj', 'Agility_adj', 'Balance_adj', 'GK Handling_adj',
       'GK Kicking_adj', 'GK Positioning_adj', 'Reactions_adj',
       'Att. Position_adj', 'Vision_adj', 'Dribbling_adj',
       'Overall Passing_adj', 'Overall Shooting_adj', 'Overall Defence_adj',
       'Overall Physical_adj', 'Overall Goalkeeper_adj']

In [93]:
# Merge merged DataFrame with fifa_df, fifa_df will be + if h_a == 'h' and - if h_a == 'a'
merged_df = pd.merge(merged_df, fifa_df, on='id', how='left')



In [94]:
merged_df[fifa_stats] = merged_df[fifa_stats].where(
    merged_df['h_a'] == 'h',
    -merged_df[fifa_stats]
)

In [95]:
# Target can be any of these
target = "PPDA"

# All inputs (choose either overalls or full FIFA features)
feature_cols = [
    # Aggregated values (example set, expand if needed)
    'Shots On Target_mean',  'DEEP_mean',
    'PPDA_median', 

    # Elo
    "ClubELO", "Seasonal_Elo",

    #  full FIFA features
    'Overall', 'Finishing_adj', 'Shot Power_adj', 'Long Shots_adj',
       'Heading_adj', 'Curve_adj', 'FK Acc._adj', 'Penalties_adj',
       'Short Pass_adj', 'Long Pass_adj', 'Stand Tackle_adj',
       'Interceptions_adj', 'Jumping_adj', 'Stamina_adj', 'Strength_adj',
       'Acceleration_adj', 'Agility_adj', 'Balance_adj', 'GK Handling_adj',
       'GK Kicking_adj', 'GK Positioning_adj', 'Reactions_adj',
       'Att. Position_adj', 'Vision_adj', 'Dribbling_adj',
       'Overall Passing_adj', 'Overall Shooting_adj', 'Overall Defence_adj',
       'Overall Physical_adj', 'Overall Goalkeeper_adj']


In [96]:
def eval_random_forest(X_train, y_train, X_test, y_test, feature_cols):
    model = RandomForestRegressor(n_estimators=300, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    importances = model.feature_importances_
    feature_importance_df = pd.DataFrame({'feature': feature_cols, 'importance': importances})
    most_important_5_list = feature_importance_df.nlargest(5, 'importance').to_dict('records')
    return mean_absolute_error(y_test, preds), r2_score(y_test, preds), most_important_5_list

def eval_neural_net(X_train, y_train, X_test, y_test):
    X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, shuffle=False)
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    model = Sequential([
        Dense(128, activation='relu', input_shape=(X_tr_scaled.shape[1],)),
        Dropout(0.2),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    es = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    model.fit(X_tr_scaled, y_tr, validation_data=(X_val_scaled, y_val),
              epochs=500, batch_size=32, callbacks=[es], verbose=0)
    
    preds = model.predict(X_test_scaled, verbose=0).ravel()
    model.scaler = scaler  # Store the scaler in the model for future use
    return mean_absolute_error(y_test, preds), r2_score(y_test, preds), None

def eval_xgboost(X_train, y_train, X_test, y_test):
    X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, shuffle=False)
    dtr = xgb.DMatrix(X_tr, label=y_tr)
    dval = xgb.DMatrix(X_val, label=y_val)
    dte  = xgb.DMatrix(X_test, label=y_test)

    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'mae',
        'max_depth': 6,
        'eta': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'seed': 42
    }
    model = xgb.train(params, dtr, num_boost_round=5000,
                      evals=[(dtr, 'train'), (dval, 'val')],
                      early_stopping_rounds=200, verbose_eval=False)
    preds = model.predict(dte, iteration_range=(0, model.best_iteration+1))
    return mean_absolute_error(y_test, preds), r2_score(y_test, preds), None

def try_models(df, feature_cols, target, full=True, n_splits=5):
    X = df[feature_cols].values
    y = df[target].values
    tscv = TimeSeriesSplit(n_splits=n_splits)

    all_results = []
    splits = list(tscv.split(X))  # cache splits

    for fold, (train_idx, test_idx) in enumerate(splits, 1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        for name, func in [
            ("Random Forest", lambda X_tr, y_tr, X_te, y_te: eval_random_forest(X_tr, y_tr, X_te, y_te, feature_cols)),
            ("Neural Network", eval_neural_net),
            ("XGBoost", eval_xgboost)
        ]:
            mae, r2, top_features = func(X_train, y_train, X_test, y_test)
            all_results.append({
                'Fold': fold,
                'Target': target,
                'Model': name,
                'MAE': mae,
                'R2': r2,
                'Feature Types': 'Full' if full else 'Overall',
                'Important Features Dictionary': top_features
            })

    results_df = pd.DataFrame(all_results)

    perf_df = results_df.groupby(
        ["Target", "Model", "Feature Types"], as_index=False
    ).agg({
        "MAE": "mean",
        "R2": "mean",
        "Important Features Dictionary": lambda x: [d for d in x if d is not None]
    })

    # Train each model on the last fold for return values
    train_idx, test_idx = splits[-1]
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Random Forest
    rf_model = RandomForestRegressor(n_estimators=300, random_state=42).fit(X_train, y_train)

    # Neural Net
    X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, shuffle=False)
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    nn_model = Sequential([
        Dense(128, activation='relu', input_shape=(X_tr_scaled.shape[1],)),
        Dropout(0.2),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])
    nn_model.compile(optimizer='adam', loss='mean_squared_error')
    nn_es = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    nn_model.fit(X_tr_scaled, y_tr, validation_data=(X_val_scaled, y_val),
                 epochs=500, batch_size=32, callbacks=[nn_es], verbose=0)
    nn_model.scaler = scaler  # Store the scaler in the model for future use

    # XGBoost
    X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, shuffle=False)
    dtr = xgb.DMatrix(X_tr, label=y_tr)
    dval = xgb.DMatrix(X_val, label=y_val)
    xgb_params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'mae',
        'max_depth': 6,
        'eta': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'seed': 42
    }
    xgb_model = xgb.train(
        xgb_params, dtr, num_boost_round=5000,
        evals=[(dtr, 'train'), (dval, 'val')],
        early_stopping_rounds=200, verbose_eval=False
    )

    return rf_model, nn_model, xgb_model, perf_df


In [97]:
performance_df = pd.DataFrame()
full_models_dict = {}

In [98]:
target_list= [
    'PPDA', 'Shots On Target', 'DEEP'
]
for target in tqdm(target_list, desc="Processing targets"):
    rf_model, nn_model, xgb_model, perf_df = try_models(
        merged_df, feature_cols, target, full=True, n_splits=5
    )
    full_models_dict[target] = {
        'Random Forest': rf_model,
        'Neural Network': nn_model,
        'XGBoost': xgb_model
    }

    performance_df = pd.concat([performance_df, perf_df], ignore_index=True)

Processing targets: 100%|██████████| 3/3 [09:04<00:00, 181.54s/it]


In [99]:
performance_df["Important Features"]= "All"

In [105]:
performance_df

,Target,Model,Feature Types,MAE,R2,Important Features Dictionary,Important Features
0,PPDA,Neural Network,Full,4.479235,0.241929,[],All
1,PPDA,Random Forest,Full,4.341606,0.275046,"[[{'feature': 'ClubELO', 'importance': 0.21541...",All
2,PPDA,XGBoost,Full,4.302734,0.263081,[],All
3,Shots On Target,Neural Network,Full,1.817102,0.111748,[],All
4,Shots On Target,Random Forest,Full,1.759312,0.178589,"[[{'feature': 'Seasonal_Elo', 'importance': 0....",All
5,Shots On Target,XGBoost,Full,1.772413,0.170143,[],All
6,DEEP,Neural Network,Full,2.959361,0.269131,[],All
7,DEEP,Random Forest,Full,2.862954,0.312075,"[[{'feature': 'DEEP_mean', 'importance': 0.253...",All
8,DEEP,XGBoost,Full,2.832105,0.323357,[],All


In [106]:
def return_best_row_for_all_targets(df, target_list):
    # Get the row with the maximum importance for the specified feature
    best_rows = pd.DataFrame()
    for target in target_list:
        best_row = df[df.Target == target].sort_values(by='MAE', ascending=True).head(1)
        best_rows = pd.concat([best_rows, best_row], ignore_index=True)
    best_rows = best_rows.reset_index(drop=True)
    return best_rows

In [107]:
return_best_row_for_all_targets(performance_df, target_list)

,Target,Model,Feature Types,MAE,R2,Important Features Dictionary,Important Features
0,PPDA,XGBoost,Full,4.302734,0.263081,[],All
1,Shots On Target,Random Forest,Full,1.759312,0.178589,"[[{'feature': 'Seasonal_Elo', 'importance': 0....",All
2,DEEP,XGBoost,Full,2.832105,0.323357,[],All


In [103]:
import numpy as np
import pandas as pd
from collections import defaultdict

def aggregate_rf_importances(perf_df, target, feature_type, model="Random Forest",
                             top_k=10, agg="sum", zero_fill_mean=False):
    """
    Aggregate per-fold RF top-k importances into one ranked table.

    agg: "sum" -> sum across folds
         "mean" -> mean across occurrences
         "mean_zero" -> mean across all folds, treating missing folds as zero
    zero_fill_mean: deprecated; use agg="mean_zero" instead
    """
    # Select the cell
    mask = (
        (perf_df['Target'] == target) &
        (perf_df['Feature Types'] == feature_type) &
        (perf_df['Model'] == model)
    )
    if not mask.any():
        raise ValueError("No matching row in perf_df for the given filters.")
    cell = perf_df.loc[mask, 'Important Features Dictionary'].values[0]

    # Count folds represented in the cell
    if isinstance(cell, list) and cell and isinstance(cell[0], list):
        n_folds = len(cell)
        nested_lists = cell
    else:
        # Already flattened to a single list (or None)
        n_folds = 1 if isinstance(cell, list) else 0
        nested_lists = [cell] if isinstance(cell, list) else []

    # Flatten to records [{'feature':..., 'importance':...}, ...]
    records = []
    for fold_list in nested_lists:
        if not isinstance(fold_list, list):
            continue
        for rec in fold_list:
            if isinstance(rec, dict) and 'feature' in rec and 'importance' in rec:
                records.append(rec)

    if not records:
        return pd.DataFrame(columns=['feature', 'score', 'occurrence', 'rank'])

    # Aggregate
    totals = defaultdict(float)
    counts = defaultdict(int)
    for rec in records:
        f = rec['feature']
        imp = float(rec['importance'])
        totals[f] += imp
        counts[f] += 1

    rows = []
    for f, total in totals.items():
        if agg == "sum":
            score = total
        elif agg == "mean":
            score = total / counts[f]
        elif agg == "mean_zero":
            # treat missing folds as zero (uses n_folds)
            score = total / max(n_folds, 1)
        else:
            raise ValueError("agg must be one of {'sum','mean','mean_zero'}")
        rows.append({'feature': f, 'score': score, 'occurrence': counts[f]})

    out = pd.DataFrame(rows).sort_values('score', ascending=False).reset_index(drop=True)
    out['rank'] = np.arange(1, len(out) + 1)
    return out.head(top_k)


In [108]:
# If your variable is named performances_df (note the plural), pass that:
for target in target_list:
    print("Target:", target, "Full Features")
    print(aggregate_rf_importances(performance_df, target=target, feature_type="Full",
                                        model="Random Forest", top_k=10, agg="sum")[["feature", "score"]])

Target: PPDA Full Features
              feature     score
0             ClubELO  1.267461
1         PPDA_median  0.448118
2         FK Acc._adj  0.181552
3       Long Pass_adj  0.095687
4         Jumping_adj  0.083144
5  GK Positioning_adj  0.062103
6    Acceleration_adj  0.048038
7         Heading_adj  0.035041
8     GK Handling_adj  0.030859
Target: Shots On Target Full Features
                feature     score
0          Seasonal_Elo  0.976031
1               ClubELO  0.312604
2           PPDA_median  0.132208
3               Overall  0.129658
4  Shots On Target_mean  0.084620
5         Reactions_adj  0.054870
6           Jumping_adj  0.034864
7         Penalties_adj  0.029256
Target: DEEP Full Features
            feature     score
0         DEEP_mean  1.041406
1           ClubELO  0.838536
2      Seasonal_Elo  0.307544
3    Short Pass_adj  0.091502
4           Overall  0.076202
5  Stand Tackle_adj  0.047873
6       Agility_adj  0.039509


In [28]:
top_features_dict = {}
top_features_dict['PPDA']=[
        "ClubELO",  #1.267461
        "PPDA_median", #0.448118
        "FK Acc._adj", #0.181552
        "Long Pass_adj", #0.095687
        "Jumping_adj", #0.083144
        "GK Positioning_adj", #0.062103
        "Acceleration_adj", #0.048038
        "Heading_adj", #0.035041
        "GK Handling_adj", #0.030859

]
top_features_dict['Shots On Target']=[
              "Seasonal_Elo", #0.976031
              "ClubELO", #0.312604
              "PPDA_median", #0.132208
              "Overall", #0.129658
              "Shots On Target_mean", #0.084620
              "Reactions_adj", #0.054870
              "Jumping_adj", #0.034864
              "Penalties_adj", #0.029256
]
top_features_dict['DEEP']=[
                "DEEP_mean", #1.041406
                "ClubELO", #0.838536
                "Seasonal_Elo", #0.307544
                "Short Pass_adj", #0.091502
                "Overall", #0.076202
                "Stand Tackle_adj", #0.047873
                "Agility_adj", #0.039509
]

In [32]:
# --- Random Forest ---
def build_RF_small(df, target, top_features, test_size=0.2):
    X = df[top_features].values
    y = df[target].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, shuffle=False
    )
    rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)
    return rf, mae, r2

# --- Neural Network (fixed: no leakage) ---
def build_NN_small(df, target, top_features, test_size=0.2, val_size=0.2):
    X = df[top_features].values
    y = df[target].values
    # Hold-out test block
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, shuffle=False
    )
    # Validation from train block (no shuffling)
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=val_size, shuffle=False
    )

    scaler = StandardScaler()
    X_tr_s   = scaler.fit_transform(X_tr)
    X_val_s  = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)

    model = Sequential([
        Dense(64, activation='relu', input_shape=(X_tr_s.shape[1],)),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    es = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

    model.fit(
        X_tr_s, y_tr,
        validation_data=(X_val_s, y_val),
        epochs=300, batch_size=32,
        callbacks=[es], verbose=0
    )

    y_pred = model.predict(X_test_s, verbose=0).ravel()
    mae = mean_absolute_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)

    # Attach scaler for downstream inference
    model.scaler = scaler
    return model, mae, r2

# --- Orchestrator ---
def get_small_models(df, top_features, target, full=True, n_splits=5):
    X = df[top_features].to_numpy()
    y = df[target].to_numpy()
    tscv = TimeSeriesSplit(n_splits=n_splits)

    perf = []
    for fold, (tr, te) in enumerate(tscv.split(X), 1):
        rf, mae_rf, r2_rf = build_RF_small(df.iloc[np.r_[tr, te]], target, top_features,
                                        test_size=len(te)/len(np.r_[tr, te]))
        nn, mae_nn, r2_nn = build_NN_small(df.iloc[np.r_[tr, te]], target, top_features,
                                        test_size=len(te)/len(np.r_[tr, te]))
        perf += [
            dict(Target=target, Model='Random Forest', MAE=mae_rf, R2=r2_rf,
                Fold=fold, Feature_Types='Full' if full else 'Overall'),
            dict(Target=target, Model='Neural Network', MAE=mae_nn, R2=r2_nn,
                Fold=fold, Feature_Types='Full' if full else 'Overall'),
        ]

    # --- average over folds & match old format ---
    perf_df = pd.DataFrame(perf)

    avg_df = (
        perf_df
        .groupby(['Target', 'Model', 'Feature_Types'], as_index=False)
        .agg(MAE=('MAE', 'mean'), R2=('R2', 'mean'))
        .rename(columns={'Feature_Types': 'Feature Types'})
    )

    # keep the same features column as before
    avg_df['Important Features'] = [top_features] * len(avg_df)

    # same columns/order as the old return
    avg_df = avg_df[['Target', 'Model', 'MAE', 'R2', 'Feature Types', 'Important Features']]

    return rf, nn, avg_df

In [35]:
full_models_dict_small = {}
for target in tqdm(target_list, desc="Processing small models"):
    top_features = top_features_dict[target]
    rf_model, nn_model, perf_df = get_small_models(
        merged_df, top_features, target, full=True
    )
    full_models_dict_small[target] = {
        'Random Forest': rf_model,
        'Neural Network': nn_model
    }
    performance_df = pd.concat([performance_df, perf_df], ignore_index=True)

Processing small models: 100%|██████████| 3/3 [02:25<00:00, 48.40s/it]


In [37]:
return_best_row_for_all_targets(performance_df, target_list)

,Target,Model,Feature Types,MAE,R2,Important Features Dictionary,Important Features
0,PPDA,XGBoost,Full,4.302734,0.263081,[],All
1,Shots On Target,Neural Network,Full,1.745202,0.183241,NaN,"[Seasonal_Elo, ClubELO, PPDA_median, Overall, ..."
2,DEEP,Neural Network,Full,2.817579,0.334113,NaN,"[DEEP_mean, ClubELO, Seasonal_Elo, Short Pass_..."


In [111]:
# --- user-provided artifacts assumed to exist ---
# merged_df
# feature_cols
# top_features_dict
# full_models_dict, full_models_dict_small

# 1) Declare which model+feature scope to use for each target (from your message)
best_plan = {
    "PPDA":                      ("XGBoost", "full"),
    "Shots On Target":           ("Neural Network", "small"),
    "DEEP":               ("Neural Network", "small"),
}


def _get_features_for(target, scope):
    if scope == "full":
        return feature_cols
    elif scope == "small":
        return top_features_dict[target]
    else:
        raise ValueError(f"Unknown scope {scope!r}")

def _get_model_for(target, model_name, scope):
    store = full_models_dict if scope == "full" else full_models_dict_small
    if target not in store or model_name not in store[target]:
        raise KeyError(f"Model for target={target}, model={model_name}, scope={scope} not found.")
    return store[target][model_name]

def _predict_with_model(model, X, feats=None):
    """Unified prediction for Keras, xgboost.Booster, sklearn-style models."""

    # Keras (Neural Network)
    if isinstance(model, tf.keras.Model):
        scaler = getattr(model, "scaler", None)
        if scaler is not None:
            X = scaler.transform(X)
        else:
            print("[WARN] Keras model has no attached scaler; using raw features.")
        return model.predict(X, verbose=0).ravel()

    # Native XGBoost Booster (from xgb.train)
    if isinstance(model, xgb.Booster):
        # If you want, you can pass feature_names=feats for extra safety:
        dX = xgb.DMatrix(X)  # or xgb.DMatrix(X, feature_names=feats)
        return model.predict(dX).ravel()

    # sklearn-style (RandomForestRegressor, XGBRegressor, etc.)
    if hasattr(model, "predict"):
        return np.asarray(model.predict(X)).ravel()

    raise TypeError(f"Unsupported model type: {type(model)}")

# --- build all {target}_agg columns ---
for target, (model_name, scope) in best_plan.items():
    feats = _get_features_for(target, scope)
    mdl   = _get_model_for(target, model_name, scope)

    # matrix in the SAME column order used in training
    X = merged_df[feats].to_numpy()

    # (optional) fail-fast NaN check
    if np.isnan(X).any():
        bad_cols = sorted(set(np.where(np.isnan(X))[1]))
        raise ValueError(
            f"NaNs in features for target '{target}': {[feats[i] for i in bad_cols]}. "
            "Impute or drop rows before predicting."
        )

    preds = _predict_with_model(mdl, X, feats)
    merged_df[f"{target}_agg"] = preds

cols_preview = [f"{t}_agg" for t in best_plan]
print("Created columns:", cols_preview)
print(merged_df[cols_preview].head())

Created columns: ['PPDA_agg', 'Shots On Target_agg', 'DEEP_agg']
    PPDA_agg  Shots On Target_agg  DEEP_agg
0  12.541602             4.390506  6.069428
1   7.177680             3.922718  8.405845
2   6.316262             5.810985  6.706576
3  10.477336             6.353431  5.967909
4   6.192487             5.424915  6.962649


In [112]:
# Round DEEP and Shots On Target to integers
merged_df['DEEP_agg'] = merged_df['DEEP_agg'].round().astype(int)
merged_df['Shots On Target_agg'] = merged_df['Shots On Target_agg'].round().astype(int)

In [113]:
merged_df.to_csv("Data/understat_team_models_aggregated.csv", index=False)

In [467]:
# 1) Enforce exact column order and numeric dtypes
X_cols = feature_cols
missing = set(X_cols) - set(merged_df.columns)
if missing:
    raise ValueError(f"Missing features in merged_df: {sorted(missing)}")

X_new = (
    merged_df
    .reindex(columns=X_cols)     # exact order used at train time
    .astype("float32")           # stable dtype for XGBoost
    .replace([np.inf, -np.inf], np.nan)  # XGB handles NaN; not inf
)

# 2) Predict with early-stopped iteration
dnew = xgb.DMatrix(X_new)
best_iter = getattr(xgb_ppda, "best_iteration", None)
if best_iter is not None:
    preds = xgb_ppda.predict(dnew, iteration_range=(0, best_iter + 1))
else:
    best_ntree_limit = getattr(xgb_ppda, "best_ntree_limit", 0)
    preds = xgb_ppda.predict(dnew, ntree_limit=best_ntree_limit) if best_ntree_limit else xgb_ppda.predict(dnew)

merged_df["PPDA_xgb_predicted"] = preds


In [468]:
# Pick the best artifact explicitly
best_model   = nn_overalls_small_deep        # trained on overalls
best_scaler  = getattr(best_model, "scaler")
best_features = top_features_overalls_deep
best_frame   = merged_overals_df

# Preconditions
missing = set(best_features) - set(best_frame.columns)
if missing:
    raise ValueError(f"Missing features for DEEP NN in merged_overals_df: {sorted(missing)}")

X_new = (best_frame
         .reindex(columns=best_features)
         .astype("float32")
         .replace([np.inf, -np.inf], np.nan))

if best_scaler is None:
    raise AttributeError("Best NN model has no attached scaler.")

# Optional safety: verify feature dimensionality before transform
if getattr(best_scaler, "n_features_in_", None) != X_new.shape[1]:
    raise ValueError(f"Feature count mismatch: X has {X_new.shape[1]} features, "
                     f"but scaler expects {best_scaler.n_features_in_}.")

# Impute with training means used by the scaler
if np.isnan(X_new.values).any():
    col_means = pd.Series(best_scaler.mean_, index=best_features)
    X_new = X_new.fillna(col_means)

X_new_s = best_scaler.transform(X_new)
best_frame["DEEP_nn_predicted"] = best_model.predict(X_new_s, verbose=0).ravel()

# If you need the prediction in merged_df too, join on a stable key
merged_df = merged_df.merge(
    best_frame[["id", "DEEP_nn_predicted" , "h_a"]],
    on=["id", "h_a"], how="left", suffixes=("", "_overalls")
)


In [442]:
best_frame.sort_values(by="id")

,id,date,team_id,Shots On Target,DEEP,PPDA,Shots On Target_mean,DEEP_mean,PPDA_median,h_a,...,Seasonal_Elo,Overall Ball Skills,Overall Passing,Overall Shooting,Overall Defence,Overall Physical,Overall Goalkeeper,Overall Mental,Overall,DEEP_nn_predicted
0,81,2015-08-08 15:45:00,89,1,4,13.8261,1.0,4.0,13.82610,h,...,57.467305,2.678571,1.809524,-1.875000,1.880952,1.500000,-0.258929,2.371429,0.955853,5.590033
3420,81,2015-08-08 15:45:00,82,4,10,8.2188,4.0,10.0,8.21880,a,...,-57.467305,-2.678571,-1.809524,1.875000,-1.880952,-1.500000,0.258929,-2.371429,-0.955853,5.046710
3421,82,2015-08-08 18:00:00,71,3,2,11.8462,3.0,2.0,11.84620,a,...,-21.339912,1.664835,2.364469,7.827610,3.970696,1.821036,-0.443681,3.985714,2.867494,4.170866
1,82,2015-08-08 18:00:00,73,2,11,6.9000,2.0,11.0,6.90000,h,...,21.339912,-1.664835,-2.364469,-7.827610,-3.970696,-1.821036,0.443681,-3.985714,-2.867494,8.067339
2,83,2015-08-08 18:00:00,72,5,5,6.6500,5.0,5.0,6.65000,h,...,39.314839,5.692308,5.723443,2.627060,-0.291209,1.861852,0.081044,4.376923,2.615750,6.805037
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3067,22303,2023-08-27 13:00:00,238,2,3,36.9091,2.0,5.0,14.30520,h,...,-162.614728,-12.988095,-14.087302,-12.440476,-10.071429,-5.848639,-1.946429,-11.907143,-9.563728,2.387801
6488,22304,2023-08-27 15:30:00,87,4,5,16.0870,5.0,11.0,7.34200,a,...,10.819436,-0.337500,0.720833,0.700000,-6.808333,2.186905,0.697917,-0.713333,-0.473363,8.629175
3068,22304,2023-08-27 15:30:00,86,8,5,10.8095,6.0,9.0,8.12915,h,...,-10.819436,0.337500,-0.720833,-0.700000,6.808333,-2.186905,-0.697917,0.713333,0.473363,6.224311
6738,25861,2024-03-13 19:30:00,256,4,1,9.1250,4.0,4.0,12.63000,a,...,-116.813483,0.226190,0.111111,-3.125000,-0.246032,3.770408,-1.303571,0.814286,-0.094482,3.764422


In [445]:
merged_df.sort_values(by="id").head(10)

,id,date,team_id,Shots On Target,DEEP,PPDA,Shots On Target_mean,DEEP_mean,PPDA_median,h_a,...,Aggression_adj,Dribbling_adj,Overall Ball Skills_adj,Overall Passing_adj,Overall Shooting_adj,Overall Defence_adj,Overall Physical_adj,Overall Goalkeeper_adj,Overall Mental_adj,DEEP_nn_predicted
0,81,2015-08-08 15:45:00,89,1,4,13.8261,1.0,4.0,13.8261,h,...,-1.607143,0.250000,1.722718,0.853671,-2.830853,0.925099,0.544147,-1.214782,1.365575,5.590033
3420,81,2015-08-08 15:45:00,82,4,10,8.2188,4.0,10.0,8.2188,a,...,1.607143,-0.250000,-1.722718,-0.853671,2.830853,-0.925099,-0.544147,1.214782,-1.365575,5.046710
3421,82,2015-08-08 18:00:00,71,3,2,11.8462,3.0,2.0,11.8462,a,...,2.076923,-3.285714,-1.202659,-0.503025,4.960116,1.103202,-1.046458,-3.311175,1.775363,4.170866
1,82,2015-08-08 18:00:00,73,2,11,6.9000,2.0,11.0,6.9000,h,...,-2.076923,3.285714,1.202659,0.503025,-4.960116,-1.103202,1.046458,3.311175,-1.775363,8.067339
2,83,2015-08-08 18:00:00,72,5,5,6.6500,5.0,5.0,6.6500,h,...,-3.723901,2.028846,3.076558,3.107693,0.011311,-2.906959,-0.753897,-2.534706,1.355404,6.805037
3422,83,2015-08-08 18:00:00,90,5,4,17.1579,5.0,4.0,17.1579,a,...,3.723901,-2.028846,-3.076558,-3.107693,-0.011311,2.906959,0.753897,2.534706,-1.355404,3.934633
3,84,2015-08-08 18:00:00,75,8,5,10.8800,8.0,5.0,10.8800,h,...,-0.847902,-1.309441,0.569431,0.621878,-1.094031,0.111389,1.373127,-1.581793,1.868382,6.064236
3423,84,2015-08-08 18:00:00,77,5,6,9.5556,5.0,6.0,9.5556,a,...,0.847902,1.309441,-0.569431,-0.621878,1.094031,-0.111389,-1.373127,1.581793,-1.868382,4.478339
4,85,2015-08-08 18:00:00,79,6,5,5.7368,6.0,5.0,5.7368,h,...,-6.782738,3.979167,-2.249232,-1.870264,1.979934,-0.775026,1.215904,1.698684,-3.234351,5.483942
3424,85,2015-08-08 18:00:00,78,7,10,10.6250,7.0,10.0,10.6250,a,...,6.782738,-3.979167,2.249232,1.870264,-1.979934,0.775026,-1.215904,-1.698684,3.234351,7.667250


In [469]:
# --- Preconditions ---
best_model    = nn_small_sot                  # NN trained on Full features
best_features = top_features_sot              # Full feature subset used in training
best_frame    = merged_df                     # Same frame family used at train time

# 1) Schema verification
missing = set(best_features) - set(best_frame.columns)
if missing:
    raise ValueError(f"Missing features for SOT NN: {sorted(missing)}")

# 2) Assemble design matrix (order + dtype + finite values)
X_new = (best_frame
         .reindex(columns=best_features)
         .astype("float32")
         .replace([np.inf, -np.inf], np.nan))

# 3) Scaler contract (must be the training scaler)
scaler = getattr(best_model, "scaler", None)
if scaler is None:
    raise AttributeError("nn_small_sot.scaler not found. Attach the training scaler before inference.")

# Optional hard check to avoid silent drift
if getattr(scaler, "n_features_in_", None) != X_new.shape[1]:
    raise ValueError(f"Feature count mismatch: X has {X_new.shape[1]}, scaler expects {scaler.n_features_in_}.")

# 4) Impute NaNs with training means (aligned to StandardScaler stats)
if np.isnan(X_new.values).any():
    col_means = pd.Series(scaler.mean_, index=best_features)
    X_new = X_new.fillna(col_means)

# 5) Transform + predict
X_new_scaled = scaler.transform(X_new)
best_frame["Shots On Target_nn_predicted"] = best_model.predict(X_new_scaled, verbose=0).ravel()


In [470]:
best_frame.columns

Index(['id', 'date', 'team_id', 'Shots On Target', 'DEEP', 'PPDA',
       'Shots On Target_mean', 'DEEP_mean', 'PPDA_median', 'h_a', 'ClubELO',
       'Seasonal_Elo', 'Overall', 'Finishing_adj', 'Shot Power_adj',
       'Long Shots_adj', 'Heading_adj', 'Curve_adj', 'FK Acc._adj',
       'Penalties_adj', 'Volleys_adj', 'Short Pass_adj', 'Long Pass_adj',
       'Crossing_adj', 'Stand Tackle_adj', 'Slide Tackle_adj',
       'Interceptions_adj', 'Jumping_adj', 'Stamina_adj', 'Strength_adj',
       'Acceleration_adj', 'Agility_adj', 'Balance_adj', 'Sprint Speed_adj',
       'GK Handling_adj', 'GK Kicking_adj', 'GK Positioning_adj',
       'GK_SaveAbility_adj', 'Reactions_adj', 'Att. Position_adj',
       'Vision_adj', 'Aggression_adj', 'Dribbling_adj',
       'Overall Ball Skills_adj', 'Overall Passing_adj',
       'Overall Shooting_adj', 'Overall Defence_adj', 'Overall Physical_adj',
       'Overall Goalkeeper_adj', 'Overall Mental_adj', 'PPDA_xgb_predicted',
       'DEEP_nn_predicted', 'S

In [474]:
# Some features should be rounded to the closest integer
best_frame["Shots On Target_nn_predicted"] = best_frame["Shots On Target_nn_predicted"].round().astype(int)
best_frame["DEEP_nn_predicted"] = best_frame["DEEP_nn_predicted"].round().astype(int)


In [475]:
best_frame

,id,date,team_id,Shots On Target,DEEP,PPDA,Shots On Target_mean,DEEP_mean,PPDA_median,h_a,...,Overall Ball Skills_adj,Overall Passing_adj,Overall Shooting_adj,Overall Defence_adj,Overall Physical_adj,Overall Goalkeeper_adj,Overall Mental_adj,PPDA_xgb_predicted,DEEP_nn_predicted,Shots On Target_nn_predicted
0,81,2015-08-08 15:45:00,89,1,4,13.8261,1.0,4.0,13.82610,h,...,1.722718,0.853671,-2.830853,0.925099,0.544147,-1.214782,1.365575,12.367002,6,4
1,82,2015-08-08 18:00:00,73,2,11,6.9000,2.0,11.0,6.90000,h,...,1.202659,0.503025,-4.960116,-1.103202,1.046458,3.311175,-1.775363,8.064672,8,4
2,83,2015-08-08 18:00:00,72,5,5,6.6500,5.0,5.0,6.65000,h,...,3.076558,3.107693,0.011311,-2.906959,-0.753897,-2.534706,1.355404,6.736794,7,5
3,84,2015-08-08 18:00:00,75,8,5,10.8800,8.0,5.0,10.88000,h,...,0.569431,0.621878,-1.094031,0.111389,1.373127,-1.581793,1.868382,10.572278,6,6
4,85,2015-08-08 18:00:00,79,6,5,5.7368,6.0,5.0,5.73680,h,...,-2.249232,-1.870264,1.979934,-0.775026,1.215904,1.698684,-3.234351,6.743678,5,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6835,22270,2024-05-19 15:00:00,71,2,0,30.3000,5.0,9.0,13.93215,a,...,-0.211687,0.115693,-0.286092,-1.939862,2.363993,-0.042045,-0.286092,10.818438,6,4
6836,22271,2024-05-19 15:00:00,229,3,1,16.6000,4.0,6.0,13.35275,a,...,-1.494544,-2.185020,-4.574901,7.672123,-0.851687,1.434028,-1.512401,21.078279,3,3
6837,22272,2024-05-19 15:00:00,228,7,5,10.5000,5.0,8.0,12.26420,a,...,0.975198,4.462378,1.282891,3.154686,-3.859966,-6.015186,0.186737,10.269228,8,5
6838,22273,2024-05-19 15:00:00,81,2,0,63.0000,4.0,4.0,14.27500,a,...,-4.778312,-2.241275,2.755876,1.650463,-3.739850,6.353098,-1.171474,44.493786,2,2


In [477]:
best_frame[best_frame["id"]==81]

,id,date,team_id,Shots On Target,DEEP,PPDA,Shots On Target_mean,DEEP_mean,PPDA_median,h_a,...,Overall Ball Skills_adj,Overall Passing_adj,Overall Shooting_adj,Overall Defence_adj,Overall Physical_adj,Overall Goalkeeper_adj,Overall Mental_adj,PPDA_xgb_predicted,DEEP_nn_predicted,Shots On Target_nn_predicted
0,81,2015-08-08 15:45:00,89,1,4,13.8261,1.0,4.0,13.8261,h,...,1.722718,0.853671,-2.830853,0.925099,0.544147,-1.214782,1.365575,12.367002,6,4
3420,81,2015-08-08 15:45:00,82,4,10,8.2188,4.0,10.0,8.2188,a,...,-1.722718,-0.853671,2.830853,-0.925099,-0.544147,1.214782,-1.365575,11.445813,5,3


In [472]:
id_list= best_frame["id"].tolist()
for i in id_list:
    if best_frame[best_frame["id"] == i].shape[0] > 2:
        print(f"Duplicate ID found: {i}")

In [476]:
best_frame["Shots On Target_nn_predicted"]

0       4
1       4
2       5
3       6
4       5
       ..
6835    4
6836    3
6837    5
6838    2
6839    6
Name: Shots On Target_nn_predicted, Length: 6840, dtype: int64

In [478]:
best_frame.to_csv("Data/team_predictions.csv", index=False)

In [214]:
performances_df.to_csv("Data/performances_for_team_aggregations.csv", index=False)